# 3.0 Batch inference — driver

Runs `legal_data_doc_parts.inference` over the corpus in Cloud Storage **inside this kernel**:
the pipeline is imported and called directly, so the config, the `Pipeline` and the returned
stats stay live objects you can inspect once a run finishes.

The same pipeline is also a CLI, and that is still the right way to run the full corpus — it
takes many hours and must not depend on this kernel staying alive:

```
python -m legal_data_doc_parts.inference --shard 0 --num-shards 1
```

For each document the pipeline reads
`document_text_parsing/info_version_9/{justice_kind}/{document_id}/paragraphs.parquet`,
classifies every paragraph, applies the sequential rule, writes `classification.parquet`
beside the input, and marks the document `is_classified` in BigQuery in batches.

**Before a real run**, confirm the GPU is alive — CPU throughput is ~23 paragraphs/sec, which
puts the full corpus at roughly a month.

In [7]:
# The pipeline is imported, not shelled out to, so an edit under inference/ is invisible
# until the module is reloaded. Autoreload must come before the imports it should track.
# (Objects built from an older definition keep it — after a dataclass field is added,
# rebuild the config rather than reusing one from an earlier cell.)
%load_ext autoreload
%autoreload 2

import logging

import pandas as pd
import torch

from legal_data_doc_parts.inference.catalog import bigquery_client
from legal_data_doc_parts.inference.config import DEFAULT_RUNS_DIR, DOCUMENT_TABLE, PipelineConfig
from legal_data_doc_parts.inference.progress import sync_from_manifest
from legal_data_doc_parts.inference.runner import Pipeline, summarise_corrections
from legal_data_doc_parts.inference.validate import report, validate

# The pipeline reports progress through logging. In a kernel the root logger has no handler,
# so without this a long run prints nothing until it returns.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-7s %(name)s | %(message)s",
    force=True,
)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: CPU only. Fix the driver before launching a full run:")
    print("  sudo modprobe -r nvidia_uvm nvidia_drm nvidia_modeset nvidia && sudo modprobe nvidia")

CUDA available: True
device: NVIDIA GeForce RTX 3070 Laptop GPU


## How much work is left

In [9]:
DOCUMENT_TABLE

'lab-test-project-1-305710.court_data_selected.document_data'

In [10]:
client = bigquery_client()
remaining = client.query(f"""
SELECT
    COUNTIF(is_parsed AND NOT is_classified) AS outstanding,
    COUNTIF(is_parsed AND is_classified)     AS classified,
    COUNTIF(is_parsed)                       AS parsed
FROM `{DOCUMENT_TABLE}`
""").to_dataframe()
remaining

/home/vegura/Projects/study/legal-data-doc-parts/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,outstanding,classified,parsed
0,4,0,4


## Calibrate

A short dry run measures real throughput without writing anywhere. Output goes under the run
directory; GCS and BigQuery are untouched (`dry_run=True` also forces `update_bigquery=False`).

In [11]:
config_test = PipelineConfig(run_id="123")

In [12]:
def run_pipeline(run_id, **overrides):
    """Run the pipeline in this kernel; returns (config, stats).

    A tqdm bar tracks documents processed out of documents selected, with live paragraph,
    skipped and failed counts; `show_progress=False` (`--no-progress`) turns it off.

    The keyword arguments are `PipelineConfig` fields — the same knobs as the CLI flags:
    `dry_run`, `limit`, `shard`, `num_shards`, `overwrite`, `retry_failed`, `batch_size`,
    `update_bigquery`, `bq_flush_size`, and `use_sequential_rule` (`--no-sequential-rule`),
    which is on by default and can be set to False to write the model's raw argmax.

    Each call builds its own `Pipeline`, so the model is loaded onto the GPU again; that is
    a few seconds, irrelevant next to any real run. Anything long belongs in a terminal —
    here a dead kernel takes the run with it, though the manifest makes it resumable.
    """
    config = PipelineConfig(run_id=run_id, runs_dir=DEFAULT_RUNS_DIR, **overrides)
    stats = Pipeline(config).run()
    print(stats.summary())
    return config, stats

In [13]:
calibration_config, calibration_stats = run_pipeline("calibration", dry_run=True, limit=50)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-09-12 05:53:10,168 INFO    legal_data_doc_parts.inference.classifier | loaded /home/vegura/Projects/study/legal-data-doc-parts/src/legal_data_doc_parts/model/checkpoint-43407 on cuda (3 labels: {0: 'introductory', 1: 'reasoning', 2: 'operative'})
/home/vegura/Projects/study/legal-data-doc-parts/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
2026-09-12 05:53:11,758 INFO    legal_data_doc_parts.inference.catalog | protected set: 0 documents (snapshotted to /home/vegura/Projects/study/legal-data-doc-parts/src/legal_data_doc_parts/runs_selected/protected_doc_ids.parquet)
/home/vegura/Projects/study/legal-data-doc-parts/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
2026-09-12 05:53:13,593 INFO    legal_data_doc_parts.infere

shard 0/1:   0%|          | 0/4 [00:00<?, ?doc/s]

2026-09-12 05:53:15,242 INFO    legal_data_doc_parts.inference.runner | done: 4 documents / 575 paragraphs in 2s (349 paragraphs/s); sequential rule changed 8 paragraphs (1.39%) across 3 documents; 0 skipped, 0 failed


4 documents / 575 paragraphs in 2s (348 paragraphs/s); sequential rule changed 8 paragraphs (1.39%) across 3 documents; 0 skipped, 0 failed


## Golden accuracy check

Runs the same inference path over documents whose Lapa-LLM labels we already have. This is
the check that catches a broken invariant — if context construction or the tokenizer pairing
regressed, agreement collapses.

Reads the local `legal_paragraphs.parquet`, so it costs nothing and touches no cloud resource.
As a CLI:

```
python -m legal_data_doc_parts.inference.validate --documents 150
```

In [14]:
golden = validate(n_documents=150)
print(report(golden))

2026-09-12 05:53:21,209 INFO    legal_data_doc_parts.inference.validate | validating on 150 documents / 5157 paragraphs


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-09-12 05:53:21,365 INFO    legal_data_doc_parts.inference.classifier | loaded /home/vegura/Projects/study/legal-data-doc-parts/src/legal_data_doc_parts/model/checkpoint-43407 on cuda (3 labels: {0: 'introductory', 1: 'reasoning', 2: 'operative'})


documents: 150   paragraphs: 5157

                        accuracy  macro-F1
raw argmax                0.9649    0.9636
+ sequential rule         0.9506    0.9491

sequential rule changed 157 paragraphs (3.04%) in 25 documents
ground truth is itself monotonic in 134/150 documents (89.3%)
  truth monotonic        n= 134 docs  raw 0.9726 -> enforced 0.9623
  truth non-monotonic    n=  16 docs  raw 0.9023 -> enforced 0.8544

              precision    recall  f1-score   support

introductory     0.9894    0.9164    0.9515      1424
   reasoning     0.9587    0.9517    0.9552      2487
   operative     0.8985    0.9872    0.9407      1246

    accuracy                         0.9506      5157
   macro avg     0.9488    0.9518    0.9491      5157
weighted avg     0.9526    0.9506    0.9507      5157



## Launch

Drop `dry_run=True` to write for real. Shard across processes or machines with
`shard=k, num_shards=N`; each shard takes `doc_id % N == k`, and they never overlap.

`use_sequential_rule=False` (CLI: `--no-sequential-rule`) writes the raw argmax instead of
forcing each document monotonic through introductory → reasoning → operative. Leave it on for
judgments — the golden check above measures what it costs and what it buys — and turn it off
to write labels the rule has not touched.

The bar counts documents against the number selected for the shard, so it is the honest
denominator for an ETA; every document ticks it once, whether it was published, skipped as
already-classified, or failed. Pass `show_progress=False` (`--no-progress`) when the output is
going to a log file rather than a terminal.

Run the full corpus from a terminal rather than the notebook — it will take many hours.

```
python -m legal_data_doc_parts.inference --run-id full-corpus --shard 0 --num-shards 4
python -m legal_data_doc_parts.inference --run-id raw-labels --no-sequential-rule
nohup python -m legal_data_doc_parts.inference --run-id full-corpus --no-progress &> run.log &
```

In [17]:
# Uncomment to launch a bounded real run in this kernel.
full_config, full_stats = run_pipeline("selected-cases-12-09", use_sequential_rule=True)

# Same run without the sequential rule, for comparing raw argmax against the enforced labels.
# raw_config, raw_stats = run_pipeline(
#     "raw-labels", dry_run=True, limit=50, use_sequential_rule=False
# )

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

2026-09-12 05:54:30,525 INFO    legal_data_doc_parts.inference.classifier | loaded /home/vegura/Projects/study/legal-data-doc-parts/src/legal_data_doc_parts/model/checkpoint-43407 on cuda (3 labels: {0: 'introductory', 1: 'reasoning', 2: 'operative'})
/home/vegura/Projects/study/legal-data-doc-parts/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
2026-09-12 05:54:32,214 INFO    legal_data_doc_parts.inference.catalog | protected set: 4 documents (snapshotted to /home/vegura/Projects/study/legal-data-doc-parts/src/legal_data_doc_parts/runs_selected/protected_doc_ids.parquet)
/home/vegura/Projects/study/legal-data-doc-parts/.venv/lib/python3.12/site-packages/google/cloud/bigquery/table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(
2026-09-12 05:54:33,925 INFO    legal_data_doc_parts.infere

0 documents / 0 paragraphs in 3s (0 paragraphs/s); sequential rule changed 0 paragraphs (0.00%) across 0 documents; 0 skipped, 0 failed


## Monitor a run

In [8]:
RUN_ID = "calibration"
config = PipelineConfig(run_id=RUN_ID, runs_dir=DEFAULT_RUNS_DIR)

if config.manifest_path.exists():
    manifest = pd.read_json(config.manifest_path, lines=True)
    print(f"{len(manifest)} documents, {manifest['n_paragraphs'].sum()} paragraphs")
    print(f"sequential rule changed {manifest['n_corrected'].sum()} paragraphs "
          f"in {int((manifest['n_corrected'] > 0).sum())} documents")
    display(manifest.tail())
else:
    print(f"no manifest yet at {config.manifest_path}")

250 documents, 6483 paragraphs
sequential rule changed 1147 paragraphs in 40 documents


,doc_id,n_paragraphs,n_corrected,seconds
245,116107613,37,0,0.005
246,116107503,24,11,0.007
247,116107601,23,0,0.005
248,116107591,62,0,0.004
249,116107593,38,0,0.001


In [9]:
if config.failures_path.exists():
    failures = pd.read_json(config.failures_path, lines=True)
    print(f"{len(failures)} failures")
    display(failures["error"].str.slice(0, 80).value_counts().head(10))
else:
    print("no failures")

no failures


## Documents the sequential rule fought with

A high correction rate means the model disagreed with the document's own structure
repeatedly. One spurious early `operative` pins the whole tail, so these are the documents
worth eyeballing.

In [10]:
if config.manifest_path.exists():
    worst = summarise_corrections(config.manifest_path)
    display(worst.head(15))
    display(worst["correction_rate"].describe())

,doc_id,n_paragraphs,n_corrected,seconds,correction_rate
49,116075950,515,460,0.003,0.893204
119,116087197,587,521,0.007,0.887564
229,116107574,51,31,0.005,0.607843
220,116107576,46,24,0.004,0.521739
246,116107503,24,11,0.007,0.458333
204,116107518,28,9,0.006,0.321429
202,116107520,28,9,0.005,0.321429
147,116098409,52,16,0.005,0.307692
28,116073689,40,12,0.006,0.300000
170,116101947,15,3,0.007,0.200000


count    250.000000
mean       0.029162
std        0.108129
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        0.893204
Name: correction_rate, dtype: float64

## Reconcile BigQuery

The manifest is the source of truth for what was written. If a run was interrupted mid-flush,
this re-marks everything it completed. Idempotent.

```
python -m legal_data_doc_parts.inference --run-id full-corpus --sync-bq
```

In [11]:
# Uncomment to reconcile is_classified for RUN_ID from its manifest.
# marked = sync_from_manifest(
#     config.manifest_path,
#     config.pending_bq_path,
#     config.bq_lock_path,
#     batch_size=config.bq_flush_size,
# )
# print(f"marked {marked} documents classified")